# Model 3 — 02 Preprocessing & Feature Schema

## Purpose
Create a deterministic, memory-conscious training cache.

The pipeline removes identifier fields such as IP addresses, ports, timestamps and Flow ID; converts traffic features to numeric values; derives binary and attack-family targets; and caps each original label so the benign majority does not overwhelm the classifier.

In [1]:

from pathlib import Path
import sys, json, os, warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "EVIDENT" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "behaviour"
MODEL_DIR = PROJECT_ROOT / "models" / "behavior"
ENV_DIR = MODEL_DIR / "environments"
META_DIR = MODEL_DIR / "metadata"
PROD_DIR = MODEL_DIR / "production"

for d in (ENV_DIR, META_DIR, PROD_DIR):
    d.mkdir(parents=True, exist_ok=True)

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Dataset folder not found: {DATA_DIR}")

import pandas as pd
import numpy as np

def clean_col(c):
    return " ".join(str(c).strip().replace("\ufeff","").split())

def norm_label(x):
    return "UNKNOWN" if pd.isna(x) else str(x).strip().upper().replace(" ","_")

def family(x):
    x = norm_label(x)
    if x == "BENIGN": return "BENIGN"
    if "PATATOR" in x or "BRUTE" in x: return "BRUTE_FORCE"
    if "PORTSCAN" in x or "PORT_SCAN" in x: return "PORT_SCAN"
    if "DDOS" in x or x.startswith("DOS"): return "DOS_DDOS"
    if "WEB_ATTACK" in x or "SQL_INJECTION" in x or "XSS" in x: return "WEB_ATTACK"
    if "BOT" in x: return "BOTNET"
    if "INFILTRATION" in x: return "INFILTRATION"
    if "HEARTBLEED" in x: return "HEARTBLEED"
    return "OTHER_ATTACK"

files = sorted(DATA_DIR.glob("*.csv"))
print("EVIDENT root:", PROJECT_ROOT)
print("CIC-IDS2017 files:", len(files))
for p in files:
    print(" -", p.name)
assert files, "No CIC-IDS2017 CSV files found."


EVIDENT root: c:\Users\bimal\OneDrive\Desktop\EVIDENT
CIC-IDS2017 files: 8
 - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
 - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
 - Friday-WorkingHours-Morning.pcap_ISCX.csv
 - Monday-WorkingHours.pcap_ISCX.csv
 - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
 - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
 - Tuesday-WorkingHours.pcap_ISCX.csv
 - Wednesday-workingHours.pcap_ISCX.csv


In [2]:

CONFIG = {
    "random_state": 42,
    "chunk_size": 100000,
    "max_rows_per_original_label": 50000,
    "drop_columns": [
        "Flow ID","Source IP","Src IP","Destination IP","Dst IP",
        "Timestamp","Source Port","Src Port","Destination Port","Dst Port","Protocol"
    ]
}

from collections import defaultdict
buffers = defaultdict(list)
counts = defaultdict(int)

for path in files:
    for chunk in pd.read_csv(path, chunksize=CONFIG["chunk_size"], low_memory=False):
        chunk.columns = [clean_col(c) for c in chunk.columns]
        chunk["Label"] = chunk["Label"].map(norm_label)
        for label, part in chunk.groupby("Label", sort=False):
            remaining = CONFIG["max_rows_per_original_label"] - counts[label]
            if remaining <= 0:
                continue
            n = min(remaining, len(part))
            buffers[label].append(part.sample(n=n, random_state=42))
            counts[label] += n

df = pd.concat(
    [pd.concat(v, ignore_index=True) for v in buffers.values()],
    ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

df["is_attack"] = (df["Label"] != "BENIGN").astype(int)
df["attack_family"] = df["Label"].map(family)

DROP = set(CONFIG["drop_columns"]) | {"Label","is_attack","attack_family"}
features = [c for c in df.columns if c not in DROP]

X = df[features].copy()
for c in features:
    X[c] = pd.to_numeric(X[c], errors="coerce")
X = X.replace([np.inf,-np.inf], np.nan)

# Remove columns that are entirely missing or constant.
features = [c for c in features if X[c].notna().any() and X[c].nunique(dropna=True) > 1]
clean = pd.concat([X[features], df[["Label","is_attack","attack_family"]]], axis=1)

out = PROJECT_ROOT/"data"/"processed"/"behavior"
out.mkdir(parents=True, exist_ok=True)
cache = out/"cicids2017_model3_training.parquet"
clean.to_parquet(cache, index=False)

schema = {
    "dataset": "CIC-IDS2017",
    "feature_columns": features,
    "feature_count": len(features),
    "dropped_identifier_columns": sorted(DROP - {"Label","is_attack","attack_family"}),
    "targets": ["is_attack","attack_family"],
    "rows": len(clean),
    "numeric_policy": "coerce numeric; inf -> NaN; median imputation inside production pipeline"
}
(META_DIR/"feature_schema.json").write_text(json.dumps(schema,indent=2),encoding="utf-8")
print("Cache:", cache)
print("Rows:", len(clean), "Features:", len(features))
print(clean["attack_family"].value_counts())


Cache: c:\Users\bimal\OneDrive\Desktop\EVIDENT\data\processed\behavior\cicids2017_model3_training.parquet
Rows: 239616 Features: 67
attack_family
DOS_DDOS        121588
PORT_SCAN        50000
BENIGN           50000
BRUTE_FORCE      15342
BOTNET            1966
WEB_ATTACK         673
INFILTRATION        36
HEARTBLEED          11
Name: count, dtype: int64
